# 🍽️ Smart Food Waste Prediction System
## LearnDepth Academy — Machine Learning Internship

> **Objective:** Build a machine learning regression model to predict the number of meals that should be prepared each day, minimizing food waste in large food service environments.

---

| Attribute | Value |
|-----------|-------|
| Dataset Size | 1,000 rows |
| Target Variable | `Meals_Consumed` |
| Best Model | Random Forest Regressor |
| Deployment | Flask Web Application |

## 📦 Step 1 — Imports & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('../dataset.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Dataset overview
print('=== Dataset Info ===')
df.info()
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Statistical Summary ===')
df.describe().round(2)

## 🔧 Step 2 — Data Preprocessing

In [ ]:
# Encode Day_of_Week as ordered numeric
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
df['Day_of_Week_Num'] = df['Day_of_Week'].apply(lambda x: day_order.index(x) if x in day_order else -1)

# One-hot encode Weather
df = pd.get_dummies(df, columns=['Weather'], prefix='Weather')

# Ensure all weather cols present
for w in ['Sunny','Cloudy','Rainy','Stormy']:
    if f'Weather_{w}' not in df.columns:
        df[f'Weather_{w}'] = 0

print('Preprocessing complete!')
print(f'Columns: {list(df.columns)}')

## 📊 Step 3 — Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Exploratory Data Analysis', fontsize=16, fontweight='bold', y=1.01)

# 1. Meals by Day
day_means = df.groupby('Day_of_Week')['Meals_Consumed'].mean()
day_means = day_means.reindex([d for d in day_order if d in day_means.index])
axes[0,0].bar(day_means.index, day_means.values, color='steelblue')
axes[0,0].set_title('Average Meals by Day of Week')
axes[0,0].set_xlabel('Day'); axes[0,0].set_ylabel('Avg Meals')
axes[0,0].tick_params(axis='x', rotation=30)

# 2. Festival Impact
df.boxplot(column='Meals_Consumed', by='Festival', ax=axes[0,1])
axes[0,1].set_title('Meals Consumed by Festival')
axes[0,1].set_xlabel('Festival (0=Normal, 1=Festival)')

# 3. Expected Customers vs Meals
axes[1,0].scatter(df['Expected_Customers'], df['Meals_Consumed'],
                  c=df['Festival'], cmap='RdYlGn', alpha=0.5, s=15)
axes[1,0].set_title('Expected Customers vs Meals Consumed')
axes[1,0].set_xlabel('Expected Customers'); axes[1,0].set_ylabel('Meals Consumed')

# 4. Weather impact
weather_cols = [c for c in df.columns if c.startswith('Weather_')]
weather_means = {c.replace('Weather_',''):df[df[c]==1]['Meals_Consumed'].mean() for c in weather_cols}
axes[1,1].bar(weather_means.keys(), weather_means.values(), color=['gold','skyblue','steelblue','gray'])
axes[1,1].set_title('Average Meals by Weather')
axes[1,1].set_xlabel('Weather'); axes[1,1].set_ylabel('Avg Meals')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation Heatmap
numeric_cols = ['Expected_Customers','Previous_Day_Consumption','Previous_Week_Same_Day',
                'Festival','Meals_Consumed']
plt.figure(figsize=(8,6))
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('Key Insight: Expected_Customers has the strongest correlation with Meals_Consumed')

## ⚙️ Step 4 — Feature Engineering

In [ ]:
# Weekend flag
df['Is_Weekend'] = df['Day_of_Week_Num'].apply(lambda x: 1 if x >= 5 else 0)

# Demand ratio: how full was previous day relative to expected
df['Demand_Ratio'] = df['Previous_Day_Consumption'] / (df['Expected_Customers'] + 1)

# Weekly trend: average of recent demand signals
df['Week_Trend'] = (df['Previous_Day_Consumption'] + df['Previous_Week_Same_Day']) / 2

# Customer efficiency from last week same day
df['Customer_Efficiency'] = df['Previous_Week_Same_Day'] / (df['Expected_Customers'] + 1)

print('New features created:')
print('  Is_Weekend       — 1 if Saturday or Sunday')
print('  Demand_Ratio     — prev day demand / expected customers')
print('  Week_Trend       — average of recent consumption signals')
print('  Customer_Efficiency — prev week same day / expected')

## ✂️ Step 5 — Train / Test Split

In [ ]:
FEATURE_COLS = [
    'Expected_Customers','Previous_Day_Consumption','Previous_Week_Same_Day',
    'Festival','Day_of_Week_Num','Is_Weekend','Demand_Ratio','Week_Trend',
    'Customer_Efficiency',
    'Weather_Sunny','Weather_Cloudy','Weather_Rainy','Weather_Stormy'
]
TARGET = 'Meals_Consumed'

X = df[FEATURE_COLS]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Training samples : {len(X_train)} ({len(X_train)/len(df)*100:.0f}%)')
print(f'Testing  samples : {len(X_test)} ({len(X_test)/len(df)*100:.0f}%)')
print(f'Features used    : {len(FEATURE_COLS)}')

## 🤖 Steps 6 & 7 — Model Selection & Training

In [ ]:
models = {
    'Random Forest': RandomForestRegressor(
        n_estimators=200, max_depth=12, min_samples_split=4, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=200, learning_rate=0.08, max_depth=5, random_state=42),
}

results = {}
for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mae  = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2   = r2_score(y_test, preds)
    results[name] = {'model': model, 'preds': preds, 'mae': mae, 'rmse': rmse, 'r2': r2}
    print(f'  MAE={mae:.2f}  RMSE={rmse:.2f}  R²={r2:.4f}\n')

## 📈 Step 8 — Model Evaluation

In [ ]:
# Summary table
summary = pd.DataFrame({
    name: {'MAE': r['mae'], 'RMSE': r['rmse'], 'R²': r['r2']}
    for name, r in results.items()
}).T.round(4)
print('=== Model Performance Summary ===')
print(summary)
best_name = summary['MAE'].idxmin()
print(f'\n🏆 Best Model: {best_name}')

In [ ]:
# Actual vs Predicted & Feature Importance
best = results[best_name]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, best['preds'], alpha=0.5, s=15, color='steelblue')
mn, mx = min(y_test.min(), best['preds'].min()), max(y_test.max(), best['preds'].max())
axes[0].plot([mn, mx], [mn, mx], 'r--', linewidth=1.5, label='Perfect prediction')
axes[0].set_title(f'{best_name}: Actual vs Predicted', fontweight='bold')
axes[0].set_xlabel('Actual'); axes[0].set_ylabel('Predicted')
axes[0].legend()

fi = pd.Series(best['model'].feature_importances_, index=FEATURE_COLS).sort_values()
axes[1].barh(fi.index, fi.values, color='steelblue')
axes[1].set_title('Feature Importance', fontweight='bold')
axes[1].set_xlabel('Importance Score')

plt.tight_layout()
plt.show()

## 💾 Step 9 — Save Model

In [ ]:
model_data = {
    'model': best['model'],
    'feature_cols': FEATURE_COLS,
    'day_order': day_order,
    'weather_options': ['Sunny','Cloudy','Rainy','Stormy'],
}
joblib.dump(model_data, '../model.pkl')
print('✅ Model saved to model.pkl')
print(f'   Model type : {type(best["model"]).__name__}')
print(f'   MAE        : {best["mae"]:.2f}')
print(f'   RMSE       : {best["rmse"]:.2f}')
print(f'   R² Score   : {best["r2"]:.4f}')

---
## ✅ Summary

| Step | Description | Status |
|------|-------------|--------|
| 1 | Data Loading | ✅ |
| 2 | Preprocessing & Encoding | ✅ |
| 3 | EDA & Visualizations | ✅ |
| 4 | Feature Engineering | ✅ |
| 5 | Train/Test Split (80/20) | ✅ |
| 6 | Model Selection | ✅ |
| 7 | Model Training | ✅ |
| 8 | Evaluation (MAE, RMSE, R²) | ✅ |
| 9 | Model Saved (Joblib) | ✅ |
| 10 | Flask Deployment | ✅ |